# 01 — Exploración de datos

**Proyecto:** Detección de fraude en transacciones bancarias (Tema 12) — IA 2026-2, UBB
**Responsable de esta parte:** Daniel Canto

**Objetivo del notebook:** conocer el dataset antes de entrenar nada: tamaño, tipos de variables,
nulos, duplicados, desbalance de clases y cómo se comportan `Amount` y `Time` en fraudes y legítimas.
Lo que se decida aquí (por ejemplo, eliminar duplicados) queda en `src/datos.py` para que el baseline y
los modelos usen exactamente los mismos datos.

**Datos:** Credit Card Fraud Detection, Machine Learning Group (ULB), Kaggle. Licencia ODbL.
Dal Pozzolo et al. (2015).

> ⚠️ Ejercicio académico: los resultados no constituyen una decisión sobre clientes reales.

## 0. Preparación del entorno

**Qué hacemos:** importamos las librerías y la configuración común del proyecto (`src/config.py`).

**Por qué:** la semilla, las rutas y el nombre de la columna objetivo se definen en un solo lugar.
Si cada notebook los escribiera a mano, podrían quedar distintos sin darnos cuenta.
Como el notebook está en `notebooks/`, agregamos la raíz del proyecto al `sys.path` para poder importar `src`.

In [ ]:
import sys
from pathlib import Path

# Buscar la raíz del proyecto (la carpeta que contiene src/) para poder importar nuestros módulos
RAIZ = Path.cwd()
while not (RAIZ / "src").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RANDOM_STATE, RUTA_CSV, DIR_CAPTURAS, COLUMNA_OBJETIVO
from src.datos import cargar_datos, eliminar_duplicados, resumen_clases

# Colores fijos para las dos clases en TODOS los gráficos (azul = legítima, naranjo = fraude)
COLORES = {0: "#2a78d6", 1: "#eb6834"}
NOMBRES = {0: "Legítima", 1: "Fraude"}

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
DIR_CAPTURAS.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ)
print("Semilla:", RANDOM_STATE)

## 1. Carga, tamaño y tipos de variables

**Qué hacemos:** cargamos el CSV **original** (todavía sin quitar duplicados, para poder analizarlos
en la sección 2) y revisamos cuántas filas y columnas tiene y de qué tipo es cada una.

**Por qué:** hay que confirmar que el archivo descargado es el esperado (284.807 filas, 31 columnas)
y que todas las variables son numéricas. Si hubiera texto o categorías, habría que codificarlas antes de
entrenar.

In [ ]:
df_original = cargar_datos(quitar_duplicados=False)

print("Filas, columnas:", df_original.shape)
df_original.head()

In [ ]:
# Tipos de dato: cuántas columnas hay de cada tipo
print(df_original.dtypes.value_counts())
print()
print("Columnas:", list(df_original.columns))

In [ ]:
# Estadísticas básicas de las columnas que SÍ son interpretables
df_original[["Time", "Amount", "Class"]].describe().round(2)

**Lectura:**
- `Time`: segundos transcurridos desde la primera transacción del dataset (cubre unos 2 días).
- `V1`–`V28`: componentes principales (PCA) que el banco entregó así por confidencialidad. **No son
  interpretables**: no sabemos qué variable original representa cada una.
- `Amount`: monto de la transacción (en euros, según la descripción del dataset).
- `Class`: 1 = fraude, 0 = legítima.

**Preguntas que nos podrían hacer:**

> **¿Qué son las columnas V1 a V28 y por qué no podemos interpretarlas?**
> Son el resultado de aplicar PCA (análisis de componentes principales) a las variables originales.
> El banco lo hizo para proteger datos confidenciales. Cada Vi es una combinación de variables originales
> que no conocemos, por eso podemos usarlas para predecir, pero no explicar qué significan.

> **¿Por qué `Time` y `Amount` no vienen transformadas como las demás?**
> Porque quedaron fuera del PCA. Por eso están en otra escala (Amount llega a miles) y hay que escalarlas
> con `StandardScaler` dentro del pipeline antes de usar modelos sensibles a la escala, como la
> regresión logística.

## 2. Valores nulos y filas duplicadas

**Qué hacemos:** contamos valores faltantes y filas repetidas, y vemos de qué clase son los duplicados.

**Por qué:** los nulos rompen el entrenamiento o obligan a imputar. Los duplicados son un problema más
sutil: si una fila queda en entrenamiento y su copia exacta en prueba, el modelo es evaluado con una
transacción que ya "memorizó" (**fuga de información**) y las métricas salen mejores de lo que son.

In [ ]:
nulos = df_original.isna().sum()
print("Total de valores nulos:", int(nulos.sum()))
print("Columnas con nulos:", list(nulos[nulos > 0].index) or "ninguna")

In [ ]:
# duplicated() marca como True las COPIAS (deja sin marcar la primera aparición)
duplicados = df_original[df_original.duplicated(keep="first")]
print("Filas duplicadas (copias):", len(duplicados))
print()
print("Clase de los duplicados:")
print(duplicados[COLUMNA_OBJETIVO].map(NOMBRES).value_counts())

# Verificación: ¿hay filas con las mismas variables pero distinta etiqueta?
# Si este número fuera mayor que el anterior, habría transacciones idénticas etiquetadas de dos formas.
dup_sin_clase = df_original.drop(columns=COLUMNA_OBJETIVO).duplicated(keep="first").sum()
print()
print("Duplicados mirando solo las variables (sin Class):", dup_sin_clase)

In [ ]:
# Eliminación con la función reutilizable de src/datos.py (la misma que usarán el baseline y los modelos)
df = eliminar_duplicados(df_original, verbose=True)

### Decisión: eliminar los duplicados antes de la partición

Según las salidas anteriores:
- El dataset no tiene valores nulos, así que no hay que imputar nada.
- Hay **1.081 filas que son copias exactas** de otra (mismas 31 columnas): **1.062 legítimas y 19 fraudes**.
- No hay transacciones idénticas con etiquetas distintas: los duplicados no son contradictorios.
- Tras eliminarlos quedan **283.726 transacciones, de las cuales 473 son fraude (0,167 %)**.
  Antes eran 492 fraudes (0,173 %).

**Decidimos eliminarlos, conservando la primera aparición, y hacerlo antes de partir en train/val/test.**

Justificación:
1. **Evitar fuga de información.** Con la partición aleatoria 70/15/15, una copia podría quedar en
   entrenamiento y otra en prueba, y el modelo sería evaluado con datos que ya vio, lo que inflaría
   la AUPRC, la precisión y el recall. Esto pesa más en los fraudes: con tan pocos casos, cada fraude repetido
   que se "filtra" a prueba cambia bastante el recall.
2. **Costo bajo.** Se pierde el 0,38 % de las filas. El desbalance casi no cambia (de 0,173 % a 0,167 %).
3. **Reproducibilidad.** La eliminación está en `src/datos.py` (`cargar_datos()` la aplica por defecto).
   El baseline, los modelos y la interfaz cargan exactamente las mismas 283.726 filas.

Limitación: no sabemos si las copias son errores de registro o transacciones reales repetidas (por ejemplo,
un doble cobro). Coincidir en las 28 componentes PCA, el monto y el segundo exacto es muy improbable
entre transacciones distintas, así que las tratamos como registros repetidos.

**A partir de aquí, todo el notebook usa `df` (sin duplicados).**

**Preguntas que nos podrían hacer:**

> **¿Por qué eliminan los duplicados antes de partir los datos y no después?**
> Si se parte primero, una fila y su copia pueden quedar en conjuntos distintos (por ejemplo, una en
> entrenamiento y otra en prueba). El modelo ya habría visto ese caso y la evaluación sería optimista.
> Eliminarlos antes garantiza que ninguna transacción aparezca en dos conjuntos.

> **¿Cuánto cambió el desbalance al quitarlos?**
> Muy poco: de 492 fraudes (0,173 %) a 473 fraudes (0,167 %) sobre 283.726 transacciones.
> El problema sigue siendo igual de desbalanceado.

## 3. Distribución de la clase objetivo (desbalance)

**Qué hacemos:** contamos fraudes y legítimas, lo graficamos y calculamos la accuracy de un
"modelo" que **nunca** predice fraude.

**Por qué:** es el centro del problema. Con un desbalance tan grande, la accuracy no sirve como métrica y
hay que usar AUPRC, precisión y recall. Este cálculo es el argumento que vamos a mostrar con el baseline
`DummyClassifier`.

In [ ]:
conteo = df[COLUMNA_OBJETIVO].value_counts().sort_index()
tabla_clases = pd.DataFrame({
    "clase": conteo.index.map(NOMBRES),
    "transacciones": conteo.values,
    "porcentaje": (100 * conteo / conteo.sum()).round(3).values,
})
print(tabla_clases.to_string(index=False))
print()
print(f"Razón: 1 fraude por cada {conteo[0] / conteo[1]:.0f} transacciones legítimas")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
barras = ax.bar(
    [NOMBRES[c] for c in conteo.index], conteo.values,
    color=[COLORES[c] for c in conteo.index], width=0.5,
)
# Escribimos el número sobre cada barra: la de fraude es casi invisible en escala lineal (ese es el punto)
for barra, valor in zip(barras, conteo.values):
    ax.annotate(f"{valor:,}".replace(",", "."), (barra.get_x() + barra.get_width() / 2, valor),
                ha="center", va="bottom", fontsize=10)
ax.set_ylim(0, conteo.max() * 1.12)  # margen arriba para que quepa la etiqueta
ax.set_ylabel("Número de transacciones")
ax.set_title("Distribución de clases (sin duplicados)")
plt.tight_layout()
plt.savefig(DIR_CAPTURAS / "01_distribucion_clases.png", dpi=150)
plt.show()

In [ ]:
# Accuracy de un clasificador que SIEMPRE dice "legítima": acierta todas las legítimas y ningún fraude
accuracy_nunca_fraude = conteo[0] / conteo.sum()
print(f"Accuracy de 'nunca fraude': {100 * accuracy_nunca_fraude:.2f}%  ->  recall de fraude = 0%")

**Preguntas que nos podrían hacer:**

> **¿Por qué no usan accuracy para evaluar el modelo?**
> Porque un modelo que nunca alerta acierta más del 99,8 % (ver salida de arriba) y no detecta
> ningún fraude. La accuracy está dominada por la clase mayoritaria. Por eso usamos AUPRC, precisión y recall,
> que se fijan en la clase fraude.

> **¿Qué consecuencias tiene el desbalance para el entrenamiento?**
> Un modelo que minimiza el error total aprende que decir siempre "legítima" casi no tiene costo.
> Por eso más adelante comparamos `class_weight="balanced"` (penalizar más los errores en fraude) y SMOTE
> (generar fraudes sintéticos, solo en entrenamiento). Además, la partición es estratificada para que
> train, val y test tengan la misma proporción de fraudes.

## 4. Distribución de `Amount` por clase

**Qué hacemos:** comparamos el monto de las transacciones legítimas y fraudulentas con estadísticas
descriptivas y un histograma.

**Por qué:** `Amount` es una de las dos variables interpretables. Queremos ver si los fraudes tienen
montos distintos y qué tan sesgada (asimétrica) es la variable, porque eso influye en el escalado.

**Dos decisiones del gráfico:**
- **Escala logarítmica** en el eje x (usamos `log10(Amount + 1)`): hay muchos montos pequeños y pocos
  muy grandes. En escala lineal todo queda aplastado contra el cero. Se suma 1 porque hay montos iguales a 0.
- **Densidad en vez de conteo** (`stat="density", common_norm=False`): cada clase se normaliza por
  separado. Con conteos, los 473 fraudes serían invisibles frente a 283 mil legítimas.

In [ ]:
desc_amount = df.groupby(COLUMNA_OBJETIVO)["Amount"].describe().rename(index=NOMBRES)
desc_amount.round(2)

In [ ]:
datos_graf = df.assign(
    log_amount=np.log10(df["Amount"] + 1),
    clase=df[COLUMNA_OBJETIVO].map(NOMBRES),
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=datos_graf, x="log_amount", hue="clase", stat="density", common_norm=False,
    bins=50, element="step", fill=False, linewidth=2,
    palette={NOMBRES[0]: COLORES[0], NOMBRES[1]: COLORES[1]}, ax=ax,
)
ax.set_xlabel("log10(Amount + 1)   [1 ≈ 9 €, 2 ≈ 99 €, 3 ≈ 999 €]")
ax.set_ylabel("Densidad (cada clase por separado)")
ax.set_title("Distribución del monto por clase")
plt.tight_layout()
plt.savefig(DIR_CAPTURAS / "01_amount_por_clase.png", dpi=150)
plt.show()

**Interpretación (TODO Daniel):** escribe aquí con tus palabras qué ves en la tabla y el gráfico.
Guía: compara las medianas (50 %) de ambas clases, el monto máximo de los fraudes frente al de las legítimas
y si hay montos 0 en fraudes. ¿Sirve `Amount` por sí sola para separar las clases?

**Preguntas que nos podrían hacer:**

> **¿Por qué usaron escala logarítmica y densidad en el histograma?**
> La escala log se usa porque `Amount` está muy sesgada a la derecha: muchos montos chicos y unos pocos
> enormes. La densidad normaliza cada clase por separado; con conteos, los fraudes no se verían.

> **¿Hay que escalar `Amount` antes de entrenar? ¿Con qué datos se ajusta el escalador?**
> Sí, para la regresión logística, porque su escala es muy distinta a la de V1–V28. Se usa `StandardScaler`
> dentro del pipeline y se ajusta **solo con entrenamiento**. Si se ajustara con todo el dataset, la media y la
> desviación de validación y prueba se filtrarían al modelo.

## 5. Distribución de `Time` por clase

**Qué hacemos:** pasamos `Time` de segundos a horas y comparamos su distribución en fraudes y legítimas.

**Por qué:** el dataset cubre unos 2 días. Si la actividad legítima sigue el ciclo día/noche y el fraude
no, la hora podría aportar información. También sirve para decidir si vale la pena transformar `Time`
(por ejemplo, a hora del día) más adelante.

In [ ]:
datos_graf["horas"] = df["Time"] / 3600
print(f"Rango de Time: {datos_graf['horas'].min():.1f} a {datos_graf['horas'].max():.1f} horas")

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=datos_graf, x="horas", hue="clase", stat="density", common_norm=False,
    bins=48, element="step", fill=False, linewidth=2,  # 48 bins = 1 bin por hora
    palette={NOMBRES[0]: COLORES[0], NOMBRES[1]: COLORES[1]}, ax=ax,
)
ax.set_xlabel("Horas desde la primera transacción")
ax.set_ylabel("Densidad (cada clase por separado)")
ax.set_title("Distribución temporal por clase")
plt.tight_layout()
plt.savefig(DIR_CAPTURAS / "01_time_por_clase.png", dpi=150)
plt.show()

**Interpretación (TODO Daniel):** ¿se ven dos "días" con valles (noches) en las legítimas? ¿Los fraudes
siguen el mismo patrón o se reparten de forma más pareja? Ojo: con solo 473 fraudes, el histograma de
fraude es ruidoso y no conviene sacar conclusiones fuertes.

**Preguntas que nos podrían hacer:**

> **¿Qué representa `Time`? ¿Es la hora del día?**
> No. Son segundos desde la primera transacción del dataset, no la hora del reloj. Para usarla como
> hora del día habría que suponer a qué hora empezó la captura, y eso no lo sabemos con certeza.

> **¿Por qué no hicieron la partición por tiempo, entrenando con el primer día y probando con el segundo?**
> Porque la propuesta comprometió una partición estratificada aleatoria 70/15/15 con semilla fija. Una
> partición temporal sería más realista para producción y la podemos mencionar como limitación o trabajo
> futuro.

## 6. Vistazo rápido a V1–V28

**Qué hacemos:** calculamos la correlación de cada componente con `Class` y la graficamos ordenada.

**Por qué:** aunque las V no son interpretables, podemos ver cuáles se relacionan más con el fraude. Esto da
una idea de si un modelo lineal (regresión logística) tiene señal con la que trabajar.

**Limitación:** la correlación de Pearson solo mide relaciones **lineales**. Una variable con correlación
baja aún puede ser útil para un modelo no lineal como Random Forest.

In [ ]:
columnas_v = [f"V{i}" for i in range(1, 29)]
corr_con_clase = df[columnas_v].corrwith(df[COLUMNA_OBJETIVO]).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(corr_con_clase.index, corr_con_clase.values, color=COLORES[0], height=0.6)
ax.axvline(0, color="gray", linewidth=1)
ax.set_xlabel("Correlación de Pearson con Class")
ax.set_title("Correlación de cada componente PCA con el fraude")
plt.tight_layout()
plt.savefig(DIR_CAPTURAS / "01_correlacion_v.png", dpi=150)
plt.show()

print("Más negativas:", list(corr_con_clase.index[:3]))
print("Más positivas:", list(corr_con_clase.index[-3:]))

**Preguntas que nos podrían hacer:**

> **Si las V no se pueden interpretar, ¿para qué sirve este gráfico?**
> Muestra que algunas componentes tienen relación lineal clara con el fraude, así que un modelo lineal
> tiene señal para aprender. También nos sirve para comparar después con la importancia de variables
> que entregue Random Forest.

> **¿Por qué las correlaciones son tan pequeñas en valor absoluto si las variables sí sirven?**
> Porque la clase fraude es muy escasa (0,167 %). Con una variable binaria tan desbalanceada la correlación
> de Pearson queda acotada lejos de 1, aunque la variable separe bien las clases. Por eso este gráfico
> sirve para comparar variables entre sí y no para medir cuánto sirve cada una por separado.

## 7. Conclusiones de la exploración (TODO Daniel)

Escribe 4–6 puntos con tus palabras, usando las salidas de arriba. Guía:
1. Tamaño final del dataset y por qué se eliminaron los duplicados.
2. Nivel de desbalance y su consecuencia para la métrica (accuracy vs AUPRC).
3. Qué se observó en `Amount` y qué implica para el escalado.
4. Qué se observó en `Time`.
5. Qué componentes V parecen más relacionadas con el fraude.
6. Próximo paso: partición estratificada 70/15/15 y baseline (`02_baseline.ipynb`).

Las figuras quedaron guardadas en `docs/capturas/` para el informe y la presentación del Avance 1.